## **1. Load the feature table and look at the three parts.**

In [2]:
import pandas as pd

df = pd.read_parquet("../data/features.parquet")

print(df.shape)
print(df.groupby("part")["is_fraud"].agg(["count", "sum"]))
print(df.groupby("part")["time"].agg(["min", "max"]))

(1852394, 29)
         count   sum
part                
test    555719  2145
train  1296675  7506
                      min                 max
part                                         
test  2020-06-21 12:14:25 2020-12-31 23:59:34
train 2019-01-01 00:00:18 2020-06-21 12:13:37


##  **2. Choose the split dates**

In [3]:
cutoff = pd.Timestamp("2020-03-21")

train_all = df[df["part"] == "train"]
fit = train_all[train_all["time"] < cutoff]
val = train_all[train_all["time"] >= cutoff]

print("learning set:  ", len(fit), "payments,", fit["is_fraud"].sum(), "fraud")
print("validation set:", len(val), "payments,", val["is_fraud"].sum(), "fraud")

learning set:   1070966 payments, 6250 fraud
validation set: 225709 payments, 1256 fraud


## **3. Make the three sets: inputs (X) and answer key (y)**

In [4]:
import sys
sys.path.append("../src")
from features import FEATURE_COLUMNS

test_all = df[df["part"] == "test"]

X_fit, y_fit = fit[FEATURE_COLUMNS], fit["is_fraud"]
X_val, y_val = val[FEATURE_COLUMNS], val["is_fraud"]
X_test, y_test = test_all[FEATURE_COLUMNS], test_all["is_fraud"]

print(len(FEATURE_COLUMNS), "features")
print("learning:  ", X_fit.shape, y_fit.shape)
print("validation:", X_val.shape, y_val.shape)
print("replay:    ", X_test.shape, y_test.shape)

24 features
learning:   (1070966, 24) (1070966,)
validation: (225709, 24) (225709,)
replay:     (555719, 24) (555719,)


## **4. Check the split**

In [5]:
print("learning:  ", fit["time"].min(), "to", fit["time"].max(), "|", int(y_fit.sum()), "fraud")
print("validation:", val["time"].min(), "to", val["time"].max(), "|", int(y_val.sum()), "fraud")
print("replay:    ", test_all["time"].min(), "to", test_all["time"].max(), "|", int(y_test.sum()), "fraud")

print("learning ends before validation starts:", fit["time"].max() < val["time"].min())
print("validation ends before replay starts:  ", val["time"].max() < test_all["time"].min())

learning:   2019-01-01 00:00:18 to 2020-03-20 23:58:44 | 6250 fraud
validation: 2020-03-21 00:00:20 to 2020-06-21 12:13:37 | 1256 fraud
replay:     2020-06-21 12:14:25 to 2020-12-31 23:59:34 | 2145 fraud
learning ends before validation starts: True
validation ends before replay starts:   True


## **Phase 4 summary**
- Learning set: Jan 2019 to 20 Mar 2020, 1,070,966 payments, 6,250 fraud
- Validation set: 21 Mar to 21 Jun 2020, 225,709 payments, 1,256 fraud
- Replay set: 21 Jun to 31 Dec 2020, 555,719 payments, 2,145 fraud
- Split by date, never at random. The replay set is not touched until the final grading.